In [70]:
import pandas as pd
import numpy as np
import ast

In [71]:
movies = pd.read_csv("../../archive/tmdb_5000_movies.csv")
credits = pd.read_csv("../../archive/tmdb_5000_credits.csv")

movies = movies.merge(credits, left_on='id', right_on='movie_id')

In [72]:
movies = movies[['id','original_title','overview','genres','keywords','cast','crew','vote_average','vote_count']]
movies = movies.dropna(subset=['overview'])

In [73]:
def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [74]:
def convert_cast(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L[:3]

movies['cast'] = movies['cast'].apply(convert_cast)

In [75]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [76]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)

In [77]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [78]:
movies['tags'] = (
    movies['overview'] +
    movies['genres'] * 3 +
    movies['keywords'] * 2 +
    movies['cast'] +
    movies['crew']
)

In [79]:
new_df = movies[['id','original_title','tags','vote_average','vote_count']]
new_df = new_df.rename(columns={'original_title':'title'})

In [80]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [81]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    return " ".join([ps.stem(word) for word in text.split()])

new_df['tags'] = new_df['tags'].apply(stem)

In [82]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
vectors = model.encode(new_df['tags'].tolist(), show_progress_bar=True)

Batches: 100%|█████████████████████████████████████████████████████████████| 150/150 [00:12<00:00, 11.81it/s]


In [83]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [84]:
new_df['normalized_rating'] = new_df['vote_average'] / 10
new_df['popularity_score'] = new_df['vote_count'] / new_df['vote_count'].max()

In [85]:
def recommend(movie):
    movie = movie.lower()
    
    if movie not in new_df['title'].str.lower().values:
        print("Movie not found")
        return
    
    movie_index = new_df[new_df['title'].str.lower() == movie].index[0]
    distances = similarity[movie_index]
    
    movies_list = list(enumerate(distances))
    
    movies_list = sorted(
        movies_list,
        reverse=True,
        key=lambda x: (
            x[1] * 0.75 +
            new_df.iloc[x[0]].normalized_rating * 0.15 +
            new_df.iloc[x[0]].popularity_score * 0.10
        )
    )[1:6]
    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [86]:
recommend("Avatar")

Interstellar
Alien
The Matrix
Battle: Los Angeles
Aliens
